# Drone Rescue v1 - DQN Training


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from drone_rescue.training.train_dqn import TrainingConfig, train_dqn


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

smoke_config = TrainingConfig(episodes=10, evaluation_interval=5, evaluation_episodes=3, device=device)
agent, smoke_history = train_dqn(smoke_config)

## Full Run

In [ ]:
config = TrainingConfig(
    episodes=500,
    size=10,
    obstacle_probability=0.2,
    seed=42,
    device=device,
    evaluation_interval=25,
    evaluation_episodes=15,
)
agent, history = train_dqn(config)

In [ ]:
history_frame = pd.DataFrame(history)
evaluation_frame = history_frame.dropna(subset=["success_rate"])

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
axes[0, 0].plot(history_frame["episode"], history_frame["reward"])
axes[0, 0].set_title("Episode Reward")
axes[0, 1].plot(evaluation_frame["episode"], evaluation_frame["success_rate"])
axes[0, 1].set_title("Evaluation Success Rate")
axes[1, 0].plot(evaluation_frame["episode"], evaluation_frame["collision_rate"])
axes[1, 0].set_title("Collision Rate")
axes[1, 1].plot(history_frame["episode"], history_frame["epsilon"])
axes[1, 1].set_title("Exploration")
plt.tight_layout()
plt.show()

In [ ]:
output_dir = Path("results/dqn_v1")
output_dir.mkdir(parents=True, exist_ok=True)
torch.save(agent.online_network.state_dict(), output_dir / "dqn.pt")
history_frame.to_csv(output_dir / "training_history.csv", index=False)
display(evaluation_frame.tail())
